# Setup

In [1]:
# set parameters
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
import plotly.graph_objects as go
import sys
sys.path.append('../../assets/python/')
import dmg5e
import estats5e
import tfb

METADATA = {'Contributor': 'T. Dunn'}
SAVEFIGS = False

In [ ]:
# functions and classes
import json
import numpy as np
import re
import uuid

book_dict = {
    "Tyranny of Dragons": {'acronym': 'ToD', 'file': 'tod.json', 'type': 'campaign', 'date': '8/19/2014'},
    "Princes of the Apocalypse": {'acronym': 'PotA', 'file': 'pota.json', 'type': 'campaign', 'date': '4/7/2015'},
    "Curse of Strahd": {'acronym': 'CoS', 'file': 'cos.json', 'type': 'campaign', 'date': '3/16/2016'},
    "Storm King's Thunder": {'acronym': 'SKT', 'file': 'skt.json', 'type': 'campaign', 'date': '9/6/2016'},
    "Tales from the Yawning Portal": {'acronym': 'TftYP', 'file': 'tftyp.json', 'type': 'anthology', 'date': '4/4/2017'},
    "Tomb of Annihilation": {'acronym': 'ToA', 'file': 'toa.json', 'type': 'campaign', 'date': '9/19/2017'},
    "Waterdeep: Dragon Heist": {'acronym': 'W:DH', 'file': 'wdh.json', 'type': 'campaign', 'date': '9/18/2018'},
    "Waterdeep: Dungeon of the Mad Mage": {'acronym': 'W:DotMM', 'file': 'wdotmm.json', 'type': 'campaign', 'date': '11/20/2018'},
    "Ghosts of Saltmarsh": {'acronym': 'GoS', 'file': 'gos.json', 'type': 'anthology', 'date': '5/21/2019'},
    "Baldur’s Gate: Descent into Avernus": {'acronym': 'BG:DiA', 'file': 'bgdia.json', 'type': 'campaign', 'date': '9/17/2019'},
    "Icewind Dale: Rime of the Frostmaiden": {'acronym': 'ID:RotF', 'file': 'idrotf.json', 'type': 'campaign', 'date': '9/15/2020'},
    "Candlekeep Mysteries": {'acronym': 'CM', 'file': 'cm.json', 'type': 'anthology', 'date': '3/16/2021'},
    "Critical Role: Call of the Netherdeep": {'acronym': 'CR:CotN', 'file': 'cotn.json', 'type': 'campaign', 'date': '3/15/2022'},
    "Journeys through the Radiant Citadel": {'acronym': 'JttRC', 'file': 'jttrc.json', 'type': 'anthology', 'date': '7/19/2022'},
    "Dragonlance: Shadow of the Dragon Queen": {'acronym': 'D:SotDQ', 'file': 'sotdq.json', 'type': 'campaign', 'date': '12/6/2022'},
    "Keys from the Golden Vault": {'acronym': 'KftGV', 'file': 'kftgv.json', 'type': 'anthology', 'date': '2/21/2023'},
    "Phandelver and Below: The Shattered Obelisk": {'acronym': 'PaB:TSO', 'file': 'pbtso.json', 'type': 'campaign', 'date': '9/19/2023'},
    "Vecna: Eve of Ruin": {'acronym': 'V:EoR', 'file': 'veor.json', 'type': 'campaign', 'date': '5/21/2024'},
    "Quests from the Infinite Staircase": {'acronym': 'QftIS', 'file': 'qftis.json', 'type': 'anthology', 'date': '7/16/2024'},
    "Dragon Delves": {'acronym': 'DD', 'file': 'drde.json', 'type': 'anthology', 'date': '7/8/2025'},
    "Forgotten Realms: Adventures in Faerûn": {'acronym': 'FR:AiF', 'file': 'fraif.json', 'type': 'anthology', 'date': '11/15/2025'},
    "Arcana Unleashed: Deadfall": {'acronym': 'AU:D', 'file': 'aud.json', 'type': 'campaign', 'date': '9/15/2026'},
}

COLOR_LIST = [
    '#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf',  # blue-teal
    # repeat
    '#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf',  # blue-teal
]

class MyEncoder(json.JSONEncoder):
    def default(self, o):
        return o.__dict__

class EncounterLibrary:
    def __init__(self, encounters=[], file=None):
        """
        Constructs a new encounter library.

        Parameters
        ----------
        encounters : list
            A list of encounters.
        
        file : str
            If provided then the encounters will be loaded from file.
        """
        if encounters:
            self.set_encounters(encounters)
        else:
            if file:
                self.from_json_file(file)

    def __repr__(self):
        return f'{self.__dict__}'

    def from_json_file(self, file):
        """
        Loads encounters from a JSON file.

        Parameters
        ----------
        file : str
            The name of the file.
        """
        with open(file, 'r') as fin:
            self.set_encounters(json.load(fin))
    
    def to_json_file(self, file, compact_lists=False, **kwargs):
        """
        Saves encounters as a JSON file.

        Parameters
        ----------
        file : str
            The name of the file.
        
        compact_lists : bool
            If ``True`` then lists will be written in a compact form with one entry per line.
            Default value ``False``.
        """
        s = json.dumps(self.encounters, cls=MyEncoder, **kwargs)
        if compact_lists:
            s = re.sub(r'(\[)[\s\n]+([^:,\[\]]+)(?=,|\])', r'\1\2', s)
            s = re.sub(r'(,)[\s\n]+([^:,\[\]]+)(?=,|\])', r'\1 \2', s)
            s = re.sub(r'(,[\s\n]+[^:,\[\]]+?)[\s\n]+(?=\])', r'\1', s)
        
        with open(file, 'w') as fout:
            fout.write(s)
    
    def get_encounters(self, book_paths=[], ids=None):
        """
        Returns a filtered list of encounters.

        Parameters
        ----------
        book_path : str
            A regular expression to be applied to encounter ``book_path`` attribute using the ``re.match`` function.
        
        Returns
        -------
        encounters : list
            The encounters that matched the given book_path regular expression.
        """    
        encounter_ids = ids if ids else []
        
        for book_path in book_paths:
            encounter_ids.extend([e.id for e in self.encounters if re.match(book_path, e.book_path)])
        
        return [e for e in self.encounters if e.id in encounter_ids]

    def set_encounters(self, encounters):
        self.encounters = []
        for e in encounters:
            if type(e) is dict:
                self.encounters.append(Encounter(**e))
            else:
                self.encounters.append(e)

class Encounter:
    def __init__(self, **kwargs):
        self.id = kwargs.get('id', str(uuid.uuid4()))
        self.type = kwargs.get('type', None)
        self.book_path = kwargs.get('book_path', None)
        self.allies = kwargs.get('allies', [])
        self.bystanders = kwargs.get('bystanders', [])
        self.enemies = kwargs.get('monsters', [])
        self.enemies = kwargs.get('enemies', self.enemies)

    def __repr__(self):
            return f'{self.__dict__}'

    def get_book(self):
        return self.book_path.split('; ')[0]

    def get_allies_xp_values(self):
        """
        Returns a list of XP values for the ally monsters in the encounter.

        Returns
        ----------
        xp_vals : list
            The XP values for each ally monster in the encounter.
        """
        xp_vals = []
        for monster in self.allies:
            xp_vals.extend(monster[0]*[monster[2]])
        
        return xp_vals

    def get_enemy_monster_ids(self):
        """
        Returns a list of monster Ids for the enemy monsters in the encounter.

        Returns
        ----------
        ids : list
            The monster ids for each enemy monster in the encounter.
        """
        ids = []
        for monster in self.enemies:
            ids.extend(monster[0]*[monster[1]])
        
        return ids
    
    def get_enemy_xp_values(self):
        """
        Returns a list of XP values for the enemy monsters in the encounter.

        Returns
        ----------
        xp_vals : list
            The XP values for each enemy monster in the encounter.
        """
        xp_vals = []
        for monster in self.enemies:
            xp_vals.extend(monster[0]*[monster[2]])
        
        return xp_vals




def player_character_xp_budget(pc_level, rules='2014'):
    """Returns the adventuring day XP budget for a single PC of the given level.
    """
    if rules == '2014':
        XP_BUDGET = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
    else:
        XP_BUDGET = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
    return XP_BUDGET[pc_level-1]

def player_character_xp_thresholds(pc_level, rules='2014'):
    """Returns the encounter XP thresholds for each encounter difficulty for a PC of the given level.
    """
    if rules == '2014':
        XP_THRESHOLDS = {
            'Trivial':[  0,  0,   0,   0,   0,   0,   0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
            'Easy':   [ 25, 50,  75, 125, 250, 300, 350, 450, 550, 600,  800, 1000, 1100, 1250, 1400, 1600, 2000, 2100, 2400, 2800], 
            'Medium': [ 50,100, 150, 250, 500, 600, 750, 900,1100,1200, 1600, 2000, 2200, 2500, 2800, 3200, 3900, 4200, 4900, 5700], 
            'Hard':   [ 75,150, 225, 375, 750, 900,1100,1400,1600,1900, 2400, 3000, 3400, 3800, 4300, 4800, 5900, 6300, 7300, 8500], 
            'Deadly': [100,200, 400, 500,1100,1400,1700,2100,2400,2800, 3600, 4500, 5100, 5700, 6400, 7200, 8800, 9500,10900,12700], 
            'Very Deadly':  [x/2 for x in [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]], 
        }
    else:
        XP_THRESHOLDS = {
            'Low':      [ 50,100,150,250, 500, 600, 750,1000,1300,1600,1900,2200,2600,2900,3300,3800, 4500, 5000, 5500, 6400],
            'Moderate': [ 75,150,225,375, 750,1000,1300,1700,2000,2300,2900,3700,4200,4900,5400,6100, 7200, 8700,10700,13200],
            'High':     [100,200,400,500,1100,1400,1700,2100,2600,3100,4100,4700,5400,6200,7800,9800,11700,14200,17200,22000],
        }
    pc_xps = {}
    for diff in XP_THRESHOLDS:
        pc_xps[diff] = XP_THRESHOLDS[diff][pc_level-1]
    return pc_xps

def party_xp_budget(levels, rules='2014'):
    """Calculate the adventuring day XP budget for a party of PCs with the given levels.
    """
    # calculates the XP budget for a party of PCs
    return sum([player_character_xp_budget(lvl, rules=rules) for lvl in levels])

def party_xp_thresholds(levels, rules='2014'):
    """calculates the XP thresholds for a party of PCs based on their levels and the rules set being used
    """
    party_xps = {}
    for lvl in levels:
        pc_xps = player_character_xp_thresholds(lvl, rules=rules)
        for diff, xp in pc_xps.items():
            party_xps[diff] = party_xps.get(diff, 0) + xp
    
    return party_xps

def encounter_multiplier_DMG(pc_count, npc_count):
    """Returns the encounter multiplier given by the 2014 DMG
    pc_count -- number of PCs in the encounter
    npc_count -- number of NPCs in the encounter
    """
    n_array = np.asarray([1,2,3,7,11,15])
    m_array = np.asarray([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0])
    i = 1 + n_array[n_array <= max(npc_count,1)].argmax()
    if pc_count >= 6:
        i -= 1
    elif pc_count <= 2:
        i += 1
    return m_array[i]

def encounter_xp_total(pc_levels, monster_xps):
    """Returns the total XP of all monsters in the encounter
    """
    return sum(monster_xps)

def encounter_adjusted_xp_total(pc_levels, monster_xps, rules='2014'):
    """Calculates the adjusted XP total for an encounter based on the number of PCs, monsters, and the monster's XP values.
    """
    xp_total = encounter_xp_total(pc_levels, monster_xps)
    if rules == '2014':
        em = encounter_multiplier_DMG(len(pc_levels), len(monster_xps))
    else:
        em = 1
    return em*xp_total

def encounter_difficulty(pc_thresholds, encounter_xp, rules='2014'):
    """Determines the encounter's difficulty category by comparing its XP value against the party's XP thresholds.
    """
    difficulties = list(pc_thresholds.keys())
    xp_values = list(pc_thresholds.values())
    indx = np.argsort(xp_values)
    
    if rules == '2014':
        difficulty = 'Trivial'
        for i in indx:
            if encounter_xp >= xp_values[i]:
                difficulty = difficulties[i]
    elif rules == '2024':
        difficulty = 'Very High'
        for i in reversed(indx):
            if encounter_xp <= xp_values[i]:
                difficulty = difficulties[i]
    return difficulty


def _find_nearest_loc(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

def monster_challenge_rating_from_xp(xp):
    id = _find_nearest_loc(dmg5e.MONSTER_DEFAULTS['XP'], xp)
    return dmg5e.MONSTER_DEFAULTS['CR'][id]

# Load Data

In [3]:
import os

encounters = []
encounters_path = '../../assets/data/encounters'
for file in os.listdir(encounters_path):
    if file == 'campaigns.json': continue
    if file.endswith('.json'):
        elib = EncounterLibrary(file=os.path.join(encounters_path, file))
        encounters.extend(elib.encounters)

elib = EncounterLibrary(encounters=encounters)
print(len(elib.encounters))

6967


In [7]:
# Construct data frame
import numpy as np
import pandas as pd

with open('../../assets/data/encounters/campaigns.json', 'r') as fin:
    campaigns = json.load(fin)

encounters = []
for campaign in campaigns:
    for group in campaign['groups']:
        encs = elib.get_encounters(ids=group['encounter_ids'])
        for enc in encs:
            if enc.type != 'combat': continue
            e = {}
            e['id'] = enc.id
            e['type'] = enc.type
            e['book'] = enc.get_book()
            e['book_acronym'] = book_dict.get(enc.get_book(), {}).get('acronym', '')
            #e['adventure'] = group['adventure']
            e['monsters'] = enc.get_enemy_monster_ids()
            e['monsters_xp'] = enc.get_enemy_xp_values()
            e['monsters_cr'] = [monster_challenge_rating_from_xp(xp) for xp in enc.get_enemy_xp_values()]
            e['n_monsters'] = len(enc.get_enemy_xp_values())
            e['n_unique_monsters'] = len(np.unique(enc.get_enemy_monster_ids()))
            e['party'] = group['party']
            e['XP_total'] = encounter_xp_total(group['party'], enc.get_enemy_xp_values())
            e['PC_XP_total'] = e['XP_total']/len(group['party'])
            try:
                e['2014 adj_XP_total'] = encounter_adjusted_xp_total(group['party'], enc.get_enemy_xp_values(), rules='2014')
                e['2014 party_xp_thresholds'] = party_xp_thresholds(group['party'], rules='2014')
                e['2014 party_xp_budget'] = party_xp_budget(group['party'], rules='2014')
                e['2014 difficulty'] = encounter_difficulty(e['2014 party_xp_thresholds'], e['2014 adj_XP_total'], rules='2014')

                e['2024 adj_XP_total'] = encounter_adjusted_xp_total(group['party'], enc.get_enemy_xp_values(), rules='2024')
                e['2024 party_xp_thresholds'] = party_xp_thresholds(group['party'], rules='2024')
                e['2024 party_xp_budget'] = party_xp_budget(group['party'], rules='2024')
                e['2024 difficulty'] = encounter_difficulty(e['2024 party_xp_thresholds'], e['2024 adj_XP_total'], rules='2024')
                encounters.append(e)
            except:
                #print(f"{enc['book_path']}:")
                #print(f"  monsters: {enc['monsters']}")
                pass

print(f'total encounters: {len(encounters)}')

df = pd.DataFrame({
    'id': [e['id'] for e in encounters],
    'book': [e['book'] for e in encounters],
    'book_acronym': [e['book_acronym'] for e in encounters],
    #'adventure': [e['adventure'] for e in encounters],
    'type': [e['type'] for e in encounters],
    'party': [e['party'] for e in encounters],
    'party_level': [np.mean(e['party']) for e in encounters],
    'monsters': [e['monsters'] for e in encounters],
    'monsters_xp': [e['monsters_xp'] for e in encounters],
    'monsters_cr': [e['monsters_cr'] for e in encounters],
    'n_monsters': [e['n_monsters'] for e in encounters],
    'n_unique_monsters': [e['n_unique_monsters'] for e in encounters],
    'XP_total': [e['XP_total'] for e in encounters],
    'PC_XP_total': [e['PC_XP_total'] for e in encounters],

    '2014 adj_XP_total': [e['2014 adj_XP_total'] for e in encounters],
    '2014 party_XP_budget': [e['2014 party_xp_budget'] for e in encounters],
    '2014 difficulty': [e['2014 difficulty'] for e in encounters],

    '2024 adj_XP_total': [e['2024 adj_XP_total'] for e in encounters],
    '2024 party_XP_budget': [e['2024 party_xp_budget'] for e in encounters],
    '2024 difficulty': [e['2024 difficulty'] for e in encounters],
})
book_categories = list(book_dict.keys())
book_acronym_categories = [v['acronym'] for v in book_dict.values()]
df['book'] = df['book'].astype('category')
df['book'] = df['book'].cat.set_categories(book_categories, ordered=True)
df['book_acronym'] = df['book_acronym'].astype('category')
df['book_acronym'] = df['book_acronym'].cat.set_categories(book_acronym_categories, ordered=True)
df['2014 difficulty'] = df['2014 difficulty'].astype('category')
df['2014 difficulty'] = df['2014 difficulty'].cat.set_categories(['Trivial','Easy','Medium','Hard', 'Deadly', 'Very Deadly'], ordered=True)
df['2014 XP_ratio'] = 2*df['2014 adj_XP_total']/df['2014 party_XP_budget']
df['2014 XP_mult'] = df['2014 adj_XP_total']/df['XP_total']
df['2024 difficulty'] = df['2024 difficulty'].astype('category')
df['2024 difficulty'] = df['2024 difficulty'].cat.set_categories(['Low','Moderate','High','Very High'], ordered=True)
df['2024 XP_ratio'] = 2*df['2024 adj_XP_total']/df['2024 party_XP_budget']
df['2024 XP_mult'] = df['2024 adj_XP_total']/df['XP_total']

total encounters: 4057


# Tables

In [ ]:
# create summary table for monsters in current dataset
import pandas as pd

books = [b for b in book_dict]
level_min = df[['book','party_level']].groupby(['book'], observed=True).min()['party_level']
level_max = df[['book','party_level']].groupby(['book'], observed=True).max()['party_level']
encounters = df[['book','id']].groupby(['book'], observed=True).count()['id']
dfT = pd.DataFrame({
    'Book': books,
    'Date': [book_dict[b]['date'] for b in books],
    'Levels': [f'{level_min[b]:.0f}-{level_max[b]:.0f}' for b in book_dict],
    'Encounters': [encounters[b] for b in book_dict],
})
dfT = dfT.astype({
    'Book': 'string', 
    'Date': 'datetime64[s]',
    #'Acronym': 'category',
    'Levels': 'string',
    'Encounters': 'int32',
})
dfT.sort_values(by='Date', inplace=True)
func = lambda s: '<a href="' + 'https://raw.githubusercontent.com/tomedunn/the-finished-book/refs/heads/master/assets/data/encounters/' + book_dict[s]['file']  + '">' + s + '</a>'
s = dfT.style.format({'Book': func, 'Date': '{:%m/%d/%Y}'}).hide(axis="index")
s.to_html(
    './encounters-table.html', 
    columns=['Book','Levels','Encounters'],
    index=False, 
    classes='wide', 
    border=0
)
dfT

,Book,Date,Levels,Encounters
0,Tyranny of Dragons,2014-08-19,1-15,198
1,Princes of the Apocalypse,2015-04-07,1-13,210
2,Curse of Strahd,2016-03-16,1-9,151
3,Storm King's Thunder,2016-09-06,1-10,225
4,Tales from the Yawning Portal,2017-04-04,1-14,521
5,Tomb of Annihilation,2017-09-19,1-11,175
6,Waterdeep: Dragon Heist,2018-09-18,1-5,161
7,Waterdeep: Dungeon of the Mad Mage,2018-11-20,5-19,438
8,Ghosts of Saltmarsh,2019-05-21,1-11,134
9,Baldur’s Gate: Descent into Avernus,2019-09-17,1-13,138
